# 🌦️ Weather Forecast - Time Series Analysis

Questo notebook guida attraverso tutte le fasi per la previsione del meteo usando tecniche di time series.

## 1. Import Required Libraries

In [ ]:
# Import delle librerie necessarie
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA

## 2. Load and Explore Weather Data

In [ ]:
# Carica i dati meteo (modifica il nome del file se necessario)
df = pd.read_csv('../data/weather.csv')
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')
print('Colonne:', df.columns.tolist())
print(df.head())
print(df.describe())
print('Valori mancanti per colonna:')
print(df.isnull().sum())

## 3. Visualize Time Series Data

In [ ]:
# Visualizza le variabili meteo nel tempo (es. temperatura, umidità)
plt.figure(figsize=(12,6))
for col in ['temperature', 'humidity', 'precipitation']:
    if col in df.columns:
        plt.plot(df.index, df[col], label=col.capitalize())
plt.legend()
plt.title('Weather Variables Over Time')
plt.xlabel('Time')
plt.ylabel('Value')
plt.show()

## 4. Preprocess Data for Forecasting

In [ ]:
# Gestione valori mancanti e creazione train/test split
df = df.fillna(method='ffill').fillna(method='bfill')
target = 'temperature' if 'temperature' in df.columns else df.columns[0]
data = df[target].values
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

## 5. Build and Train Time Series Forecasting Model

In [ ]:
# Modello ARIMA per previsione della variabile target
model = ARIMA(train, order=(3,1,2))
model_fit = model.fit()
preds = model_fit.forecast(steps=len(test))

## 6. Evaluate Model Performance

In [ ]:
# Valutazione delle performance del modello
mae = mean_absolute_error(test, preds)
rmse = np.sqrt(mean_squared_error(test, preds))
print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
plt.figure(figsize=(12,6))
plt.plot(range(len(train)), train, label='Train')
plt.plot(range(len(train), len(train)+len(test)), test, label='Test')
plt.plot(range(len(train), len(train)+len(test)), preds, label='Predictions')
plt.legend()
plt.title('ARIMA Forecast vs Actual')
plt.xlabel('Time')
plt.ylabel(target.capitalize())
plt.show()

## 7. Make Future Weather Predictions

In [ ]:
# Previsione dei valori futuri (es. prossimi 7 giorni)
future_steps = 7
model_full = ARIMA(data, order=(3,1,2)).fit()
future_preds = model_full.forecast(steps=future_steps)
print('Previsioni per i prossimi giorni:', future_preds)
plt.figure(figsize=(12,6))
plt.plot(df.index, data, label='Storico')
future_index = pd.date_range(df.index[-1], periods=future_steps+1, freq='D')[1:] if hasattr(df.index, 'freq') or isinstance(df.index, pd.DatetimeIndex) else range(len(data), len(data)+future_steps)
plt.plot(future_index, future_preds, label='Previsioni Future')
plt.legend()
plt.title('Previsioni Meteo Future')
plt.xlabel('Time')
plt.ylabel(target.capitalize())
plt.show()

---

**Istruzioni rapide:**
- Carica il tuo file CSV meteo in `../data/weather.csv`.
- Assicurati che abbia almeno una colonna temporale (`date` o simile) e una variabile target (`temperature`, `humidity`, ecc.).
- Se la colonna temporale non è indicata, modifica la cella di caricamento dati per impostare l'indice temporale.
- Il notebook usa ARIMA per la previsione della variabile target.

**Se vuoi un modello LSTM o Prophet, chiedi!**